In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

In [3]:
df = pd.read_csv("data/retail_store_sales.csv")

## 1. Initial data inspection

In [4]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.9+ MB


In [6]:
df.describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [7]:
df.describe(include=["str", "object"])

,Transaction ID,Customer ID,Category,Item,Payment Method,Location,Transaction Date,Discount Applied
count,12575,12575,12575,11362,12575,12575,12575,8376
unique,12575,25,8,200,3,2,1114,2
top,TXN_6867343,CUST_05,Furniture,Item_2_BEV,Cash,Online,2022-05-30,True
freq,1,544,1591,126,4310,6354,26,4219


## 2. Correct columns data types

In [8]:
df.dtypes

Transaction ID          str
Customer ID             str
Category                str
Item                    str
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
Discount Applied     object
dtype: object

In [9]:
df["Quantity"].unique()

array([10.,  9.,  2.,  7.,  8., nan,  1.,  3.,  6.,  4.,  5.])

In [10]:
df["Quantity"] = df["Quantity"].astype("Int64")

In [11]:
df["Transaction Date"].head()

0    2024-04-08
1    2023-07-23
2    2022-10-05
3    2022-05-07
4    2022-10-02
Name: Transaction Date, dtype: str

In [12]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], format="%Y-%m-%d")

In [13]:
df["Discount Applied"].unique()

array([True, False, nan], dtype=object)

In [14]:
df["Discount Applied"] = df["Discount Applied"].astype("boolean")

In [15]:
df.dtypes

Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                     Int64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied           boolean
dtype: object

## 3. Handling missing values

In [16]:
cols_with_missing = (df.isna().mean() * 100).round(2)
cols_with_missing[cols_with_missing > 0]

Item                 9.65
Price Per Unit       4.84
Quantity             4.80
Total Spent          4.80
Discount Applied    33.39
dtype: float64

In [17]:
df["Discount Applied"].value_counts(dropna=False)

Discount Applied
True     4219
<NA>     4199
False    4157
Name: count, dtype: Int64

In [18]:
df["Discount Applied"] = df["Discount Applied"].astype("str").fillna("Unknown")

In [19]:
cols_with_missing = (df.isna().mean() * 100).round(2)
cols_with_missing[cols_with_missing > 0]

Item              9.65
Price Per Unit    4.84
Quantity          4.80
Total Spent       4.80
dtype: float64

In [20]:
df[df["Price Per Unit"].notna() & df["Quantity"].notna()]

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4,152.0,Credit Card,In-store,2023-09-03,Unknown
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10,140.0,Cash,Online,2024-08-24,Unknown
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6,84.0,Cash,Online,2023-12-30,True


In [21]:
# Count rows where 'Price Per Unit' and 'Quantity' have values, while 'Total Spent' is missing
count = (
    df["Price Per Unit"].notna() & df["Quantity"].notna() & df["Total Spent"].isna()
).sum()
print(count)

0


In [22]:
# Count rows where 'Total Spent' and 'Quantity' have values, while 'Price Per Unit' is missing
count = (
    df["Total Spent"].notna() & df["Quantity"].notna() & df["Price Per Unit"].isna()
).sum()
print(count)

609


In [23]:
# Fill missing 'Price Per Unit' using (Total Spent / Quantity)
mask = df["Total Spent"].notna() & df["Quantity"].notna() & df["Price Per Unit"].isna()

df.loc[mask, "Price Per Unit"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Quantity"]

In [24]:
# CHECK IF WORKED: Count rows where 'Total Spent' and 'Quantity' have values, while 'Price Per Unit' is missing
count = (
    df["Total Spent"].notna() & df["Quantity"].notna() & df["Price Per Unit"].isna()
).sum()
print(count)

0


In [25]:
# Count rows where 'Price Per Unit' and 'Total Spent' have values, while 'Quantity' is missing
count = (
    df["Price Per Unit"].notna() & df["Total Spent"].notna() & df["Quantity"].isna()
).sum()
print(count)

0


In [26]:
# Count rows where 'Total Spent' has values, while 'Quantity' and 'Price Per Unit' are missing
count = (
    df["Total Spent"].notna() & df["Price Per Unit"].isna() & df["Quantity"].isna()
).sum()
print(count)

0


In [27]:
# Count rows where 'Quantity' has values, while 'Total Spent' and 'Price Per Unit' are missing
count = (
    df["Quantity"].notna() & df["Total Spent"].isna() & df["Price Per Unit"].isna()
).sum()
print(count)

0


In [28]:
# Count rows where 'Price Per Unit' has values, while 'Total Spent' and 'Quantity' are missing
count = (
    df["Price Per Unit"].notna() & df["Total Spent"].isna() & df["Quantity"].isna()
).sum()
print(count)

604


In [29]:
mask = df["Price Per Unit"].notna() & df["Total Spent"].isna() & df["Quantity"].isna()

df.loc[mask, ["Item", "Price Per Unit", "Quantity", "Total Spent"]]

,Item,Price Per Unit,Quantity,Total Spent
7,NaN,33.5,<NA>,NaN
15,NaN,24.5,<NA>,NaN
19,NaN,35.0,<NA>,NaN
25,NaN,39.5,<NA>,NaN
34,NaN,23.0,<NA>,NaN
...,...,...,...,...
12527,NaN,5.0,<NA>,NaN
12552,NaN,8.0,<NA>,NaN
12556,NaN,41.0,<NA>,NaN
12562,NaN,33.5,<NA>,NaN


In [30]:
mask = df["Price Per Unit"].notna() & df["Total Spent"].isna() & df["Quantity"].isna()

df.loc[mask, ["Category", "Item", "Price Per Unit", "Quantity", "Total Spent"]]

,Category,Item,Price Per Unit,Quantity,Total Spent
7,Furniture,NaN,33.5,<NA>,NaN
15,Beverages,NaN,24.5,<NA>,NaN
19,Furniture,NaN,35.0,<NA>,NaN
25,Furniture,NaN,39.5,<NA>,NaN
34,Patisserie,NaN,23.0,<NA>,NaN
...,...,...,...,...,...
12527,Food,NaN,5.0,<NA>,NaN
12552,Milk Products,NaN,8.0,<NA>,NaN
12556,Beverages,NaN,41.0,<NA>,NaN
12562,Butchers,NaN,33.5,<NA>,NaN


In [31]:
check_item_price = (
    df[df["Item"].notna() & df["Price Per Unit"].notna()]
    .groupby("Item")["Price Per Unit"]
    .nunique()
)

check_item_price[check_item_price > 1]

Series([], Name: Price Per Unit, dtype: int64)

In [32]:
item_category_check = (
    df[df["Item"].notna() & df["Category"].notna()]
    .groupby("Item")["Category"]
    .nunique()
)

item_category_check[item_category_check > 1]

Series([], Name: Category, dtype: int64)

In [33]:
ref_df = (
    df[["Category", "Item", "Price Per Unit"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["Category", "Item"])
    .reset_index(drop=True)
)

ref_df

,Category,Item,Price Per Unit
0,Beverages,Item_10_BEV,18.5
1,Beverages,Item_11_BEV,20.0
2,Beverages,Item_12_BEV,21.5
3,Beverages,Item_13_BEV,23.0
4,Beverages,Item_14_BEV,24.5
...,...,...,...
195,Patisserie,Item_5_PAT,11.0
196,Patisserie,Item_6_PAT,12.5
197,Patisserie,Item_7_PAT,14.0
198,Patisserie,Item_8_PAT,15.5


In [34]:
count = (df["Item"].isna() & df["Price Per Unit"].notna()).sum()
print(count)

1213


In [35]:
mask = df["Item"].isna() & df["Category"].notna() & df["Price Per Unit"].notna()

df.loc[mask, ["Category", "Item", "Price Per Unit"]]

,Category,Item,Price Per Unit
5,Patisserie,NaN,20.0
7,Furniture,NaN,33.5
11,Milk Products,NaN,6.5
15,Beverages,NaN,24.5
17,Milk Products,NaN,27.5
...,...,...,...
12527,Food,NaN,5.0
12552,Milk Products,NaN,8.0
12556,Beverages,NaN,41.0
12562,Butchers,NaN,33.5


In [36]:
for index in df[mask].index:

    category = df.loc[index, "Category"]
    price = df.loc[index, "Price Per Unit"]

    item = ref_df[
        (ref_df["Category"] == category) & (ref_df["Price Per Unit"] == price)
    ]["Item"].iloc[0]

    df.loc[index, "Item"] = item

In [37]:
cols_with_missing = (df.isna().mean() * 100).round(2)
cols_with_missing[cols_with_missing > 0]

Quantity       4.8
Total Spent    4.8
dtype: float64

In [38]:
percentage = ((df["Quantity"].isna() & df["Total Spent"].isna()).mean() * 100).round(2)

print(percentage)

4.8


In [39]:
df = df.dropna(subset=["Quantity", "Total Spent"], how="all")

In [40]:
cols_with_missing = (df.isna().mean() * 100).round(2)
cols_with_missing[cols_with_missing > 0]

Series([], dtype: float64)

In [41]:
df.duplicated().sum()

np.int64(0)

In [42]:
df.reset_index(drop=True)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
11966,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4,152.0,Credit Card,In-store,2023-09-03,Unknown
11967,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9,58.5,Cash,Online,2022-08-12,False
11968,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10,140.0,Cash,Online,2024-08-24,Unknown
11969,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6,84.0,Cash,Online,2023-12-30,True


## 4. Feature Engineering

In [43]:
df["Transaction Date"].unique()

<DatetimeArray>
['2024-04-08 00:00:00', '2023-07-23 00:00:00', '2022-10-05 00:00:00',
 '2022-05-07 00:00:00', '2022-10-02 00:00:00', '2023-11-30 00:00:00',
 '2023-06-10 00:00:00', '2023-04-26 00:00:00', '2024-03-14 00:00:00',
 '2024-12-14 00:00:00',
 ...
 '2023-03-28 00:00:00', '2023-03-16 00:00:00', '2022-07-18 00:00:00',
 '2022-10-20 00:00:00', '2022-11-06 00:00:00', '2022-08-28 00:00:00',
 '2023-06-06 00:00:00', '2024-02-29 00:00:00', '2022-09-16 00:00:00',
 '2023-03-17 00:00:00']
Length: 1114, dtype: datetime64[us]

In [44]:
df["Transaction Year"] = df["Transaction Date"].dt.year
df["Transaction Month"] = df["Transaction Date"].dt.month
df["Transaction Day"] = df["Transaction Date"].dt.day

df.drop(columns=["Transaction Date"], inplace=True)
df.drop(columns=["Transaction ID"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        11971 non-null  str    
 1   Category           11971 non-null  str    
 2   Item               11971 non-null  str    
 3   Price Per Unit     11971 non-null  float64
 4   Quantity           11971 non-null  Int64  
 5   Total Spent        11971 non-null  float64
 6   Payment Method     11971 non-null  str    
 7   Location           11971 non-null  str    
 8   Discount Applied   11971 non-null  str    
 9   Transaction Year   11971 non-null  int32  
 10  Transaction Month  11971 non-null  int32  
 11  Transaction Day    11971 non-null  int32  
dtypes: Int64(1), float64(2), int32(3), str(6)
memory usage: 1.6 MB


In [46]:
df.to_csv("./data/cleaned_df.csv", index=False)

## EDA

In [47]:
df = pd.read_csv("./data/cleaned_df.csv")

In [48]:
df.head()

,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Discount Applied,Transaction Year,Transaction Month,Transaction Day
0,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,True,2024,4,8
1,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,True,2023,7,23
2,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,False,2022,10,5
3,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,Unknown,2022,5,7
4,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,False,2022,10,2


In [49]:
numeric_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(include=["str", "object"]).columns

In [ ]:
# Q. Which category has the highest total sales?
"""
Insight...
Total sales are fairly balanced across categories, with 'Butchers' leading and 'Milk Products' recording the lowest sales.
"""

categories_sales = (
    df.groupby("Category")["Total Spent"].sum().sort_values(ascending=False)
)

fig = px.bar(
    categories_sales,
    title="Total Sales by Category",
    labels={"value": "Total Sales"},
    text=categories_sales.values,
)

fig.show()

In [ ]:
# Q. Which payment method generates the highest total sales?
"""
Insight...
Cash leads total sales, but sales are relatively evenly distributed across all three payment methods.
"""

payment_sales = (
    df.groupby("Payment Method")["Total Spent"].sum().sort_values(ascending=False)
)
fig = px.bar(
    payment_sales,
    title="Total Sales by Payment Method",
    text=payment_sales.values,
    labels={"value": "Total Sales"},
)
fig.show()

In [133]:
# Q. Which location generates the highest total sales?
"""
Insight...
Online sales are slightly higher than in-store sales, indicating a relatively balanced sales contribution across locations.
"""

location_sales = (
    df.groupby("Location")["Total Spent"].sum().sort_values(ascending=False)
)

fig = px.bar(
    location_sales,
    title="Total Sales by Location",
    labels={"value": "Total Sales"},
    text=location_sales.values,
)

fig.show()

In [134]:
# Q. Which category has the highest total quantity sold?
"""
Insight...
Furniture has the highest quantity sold, while Patisserie records the lowest, with sales volume relatively balanced across categories.
"""

categories_quantity = (
    df.groupby("Category")["Quantity"].sum().sort_values(ascending=False)
)
fig = px.bar(
    categories_quantity,
    title="Total Quantity Sold by Category",
    labels={"value": "Total Quantity"},
    text=categories_quantity.values,
)

fig.show()

In [138]:
# Q. What is the average amount spent per transaction in each category?
"""
Insight...
Butchers has the highest average spending per transaction, while Milk Products has the lowest, with relatively close averages across categories.
"""

category_avg_spent = (
    df.groupby("Category")["Total Spent"].mean().sort_values(ascending=False)
)
fig = px.bar(
    category_avg_spent,
    title="Average Spent per Transaction by Category",
    labels={"value": "Average of Total Spent"},
    text=category_avg_spent.values.round(2),
)

fig.show()

In [139]:
# Q. Is there a relationship between the price per unit and the quantity purchased?
"""
Insight...
There is no relationship between Price Per Unit and Quantity purchased
"""

fig = px.scatter(
    df, x="Price Per Unit", y="Quantity", title="Price Per Unit vs Quantity"
)

fig.show()

In [142]:
# Q. How did total sales change across the transaction years?
"""
Insight...
Total sales fluctuated across the years, peaking in 2024 before dropping sharply in 2025.
"""

yearly_sales = df.groupby("Transaction Year", as_index=False)["Total Spent"].sum()
yearly_sales["Transaction Year"] = yearly_sales["Transaction Year"].astype(str)

fig = px.line(
    yearly_sales,
    x="Transaction Year",
    y="Total Spent",
    title="Total Sales by Year",
    markers=True,
)

fig.show()

In [145]:
# Q. Which items are the most frequently purchased?
"""
Insights...
Item_2_BEV is the most frequently purchased item, while the top purchases are distributed across several product categories.
"""

item_counts = df["Item"].value_counts().head(10)
fig = px.bar(
    x=item_counts.index,
    y=item_counts.values,
    labels={"x": "Item", "y": "Number of Transactions"},
    title="Top 10 Most Frequently Purchased Items",
    text=item_counts.values,
)

fig.show()

In [ ]:
# Q. Which payment method is most commonly used?
"""
Insights...
Cash is the most commonly used payment method, while digital wallets and credit cards show relatively similar usage.
"""

payment_counts = df["Payment Method"].value_counts()
fig = px.bar(
    x=payment_counts.index,
    y=payment_counts.values,
    text=payment_counts.values,
    labels={"x": "Payment Method", "y": "Number of Transactions"},
    title="Number of Transactions by Payment Method",
)

fig.show()

In [150]:
# Q. Which category has the highest average price per unit?
"""
Insights...
Butchers has the highest average price per unit, while Milk Products has the lowest, with most categories showing relatively similar average prices.
"""

avg_price = df.groupby("Category")["Price Per Unit"].mean().sort_values(ascending=False)
fig = px.bar(
    x=avg_price.index,
    y=avg_price.values,
    text=avg_price.values.round(2),
    labels={"x": "Category", "y": "Average Price Per Unit"},
    title="Average Price per Unit by Category",
)

fig.show()

In [ ]:
# Q. Which month has the highest number of transactions?
"""
Insights...
January has the highest number of transactions, while February has the lowest, with transaction activity varying across the months.
"""

monthly_transactions = df["Transaction Month"].value_counts()
fig = px.bar(
    x=monthly_transactions.index,
    y=monthly_transactions.values,
    text=monthly_transactions.values,
    labels={"x": "Month", "y": "Number of Transactions"},
    title="Transactions by Month",
)

fig.show()

In [ ]:
# Q. Which items generate the highest total sales?
"""
Insights...
Item_25_FUR generates the highest total sales, with several top-performing items coming from Furniture and other categories.
"""

item_sales = (
    df.groupby("Item")["Total Spent"].sum().sort_values(ascending=False).head(10)
)
fig = px.bar(
    x=item_sales.index,
    y=item_sales.values,
    text=item_sales.values.round(2),
    labels={"x": "Item", "y": "Total Sales"},
    title="Top 10 Items by Total Sales",
)

fig.show()

In [157]:
# Q. Which discount status generates the highest total sales?
"""
Insights...
Transactions with discounts generate the highest total sales, while sales remain relatively close across the different discount statuses.
"""

sum_spent_discount = (
    df.groupby("Discount Applied")["Total Spent"].sum().sort_values(ascending=False)
)
fig = px.bar(
    x=sum_spent_discount.index,
    y=sum_spent_discount.values,
    text=sum_spent_discount.values,
    labels={"x": "Discount Applied", "y": "Sum Total Spent"},
    title="Sum of Total Spent by Discount Status",
)

fig.show()

In [160]:
# Q. Which category has the highest number of unique customers?
"""
Insights...
All categories have the same number of unique customers, indicating an evenly distributed customer base across categories.
"""

unique_customers = (
    df.groupby("Category")["Customer ID"].nunique().sort_values(ascending=False)
)
fig = px.bar(
    x=unique_customers.index,
    y=unique_customers.values,
    text=unique_customers.values,
    labels={"x": "Category", "y": "Number of Unique Customers"},
    title="Unique Customers by Category",
)

fig.show()

In [167]:
# Q. How does total sales vary by transaction month?
"""
Insights...
January had the highest total sales, while October had the lowest. Sales changed across the months, with no clear pattern.
"""

monthly_sales = (
    df.groupby("Transaction Month")["Total Spent"].sum().sort_values(ascending=False)
)
fig = px.bar(
    x=monthly_sales.index.astype(str),
    y=monthly_sales.values,
    text=monthly_sales.values.round(2),
    labels={"x": "Transaction Month", "y": "Total Sales"},
    title="Total Sales by Transaction Month",
)

fig.show()

In [ ]:
# Q. Which category has the highest average quantity per transaction?
"""
Insights...
Computers and electric accessories has the highest average quantity per transaction, while electric household essentials has the lowest. The average quantity is quite similar across all categories.
"""

avg_quantity_category = (
    df.groupby("Category")["Quantity"].mean().sort_values(ascending=False)
)
fig = px.bar(
    x=avg_quantity_category.index,
    y=avg_quantity_category.values,
    text=avg_quantity_category.values.round(2),
    labels={"x": "Category", "y": "Average Quantity"},
    title="Average Quantity per Transaction by Category",
)

fig.show()

In [171]:
# Q. Which customers have the highest total spending?
"""
Insights...
CUST_24 has the highest total spending, followed by CUST_08 and CUST_05. These customers are the top spenders in the dataset.
"""

customer_spending = (
    df.groupby("Customer ID")["Total Spent"].sum().sort_values(ascending=False).head(10)
)
fig = px.bar(
    x=customer_spending.index,
    y=customer_spending.values,
    text=customer_spending.values.round(2),
    labels={"x": "Customer ID", "y": "Total Spent"},
    title="Top 10 Customers by Total Spending",
)

fig.show()

In [181]:
# Q. How many transactions does each customer make?
"""
Insights...
CUST_24 makes the most transactions, while CUST_10 and CUST_23 make the fewest. Overall, transaction counts are fairly similar across the top 10 customers.
"""

customer_transactions = df["Customer ID"].value_counts().head(10)
fig = px.bar(
    x=customer_transactions.index,
    y=customer_transactions.values,
    text=customer_transactions.values,
    labels={"x": "Customer ID", "y": "Number of Transactions"},
    title="Top 10 Customers by Number of Transactions",
)

fig.show()

In [178]:
# What is the distribution of transaction amounts?
"""
Insights...
Almost the most of transaction amount under 200 `Right Skewed`
"""

fig = px.histogram(
    df,
    x="Total Spent",
    nbins=30,
    labels={"Total Spent": "Transaction Amount"},
    title="Distribution of Transaction Amounts",
)

fig.show()

In [177]:
# Q. What is the distribution of quantities purchased?
"""
Insights...
There is no pattern at the distribution of quantities purchased
"""

fig = px.histogram(
    df,
    x="Quantity",
    nbins=10,
    labels={"Quantity": "Quantity Purchased"},
    title="Distribution of Quantity Purchased",
)

fig.show()

In [176]:
# Q. Which category has the highest number of transactions?
"""
Insights...
Furniture has the highest number of transactions, while Patisserie has the lowest. Overall, transactions are fairly evenly distributed across categories.
"""

category_transactions = df["Category"].value_counts().sort_values(ascending=False)
fig = px.bar(
    x=category_transactions.index,
    y=category_transactions.values,
    text=category_transactions.values,
    labels={"x": "Category", "y": "Number of Transactions"},
    title="Number of Transactions by Category",
)

fig.show()

In [ ]:
# Q. What is the distribution of the number of transcations across years?
"""
Insights...
Transaction counts were relatively stable from 2022 to 2024, with 2024 having the highest number of transactions. 2025 shows a sharp drop in transactions.
"""

year_transactions = df["Transaction Year"].value_counts().sort_index()
fig = px.line(
    x=year_transactions.index.astype(str),
    y=year_transactions.values,
    markers=True,
    labels={"x": "Transaction Year", "y": "Number of Transactions"},
    title="Number of Transactions Across Years",
)

fig.show()